# SAR Automatic Target Recognition — Full Pipeline

**Environment**: Vast.ai GPU instance  
**Dataset**: MSTAR (10 vehicle classes, ~1000 SAR images)  
**Model**: CNN + GNN few-shot open-set recognition  

Run cells top-to-bottom. Each section is self-contained and can be re-run independently.

---

## 0. GPU Check

In [1]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — check instance type')

Sun Apr 12 17:07:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.133.07             Driver Version: 570.133.07     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5080        On  |   00000000:02:00.0 Off |                  N/A |
|  0%   44C    P8             27W /  360W |       0MiB /  16303MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))
    print('VRAM            :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU             : NVIDIA GeForce RTX 5080
VRAM            : 16.6 GB


## 1. Environment Setup

**Why `sys.executable` instead of `!pip`?**  
On Vast.ai, the Jupyter kernel runs inside a specific Python environment (e.g. a conda env or virtualenv). The bare `!pip` command calls whatever `pip` is first on the system `PATH` — which is often a *different* Python than the kernel. Packages installed there are invisible to the kernel, causing `ModuleNotFoundError` even after a successful install. Using `sys.executable -m pip` guarantees the install goes into the exact Python that is running this notebook.

In [3]:
import sys

# Show exactly which Python the kernel is using
print('Kernel Python   :', sys.executable)
print('Python version  :', sys.version)
print()
print('This is the Python that pip must install into.')
print('The next cell uses sys.executable to guarantee that.')

Kernel Python   : /venv/main/bin/python
Python version  : 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:50:00) [GCC 14.3.0]

This is the Python that pip must install into.
The next cell uses sys.executable to guarantee that.


In [4]:
import sys

# Install all packages into the kernel's own Python — never fails due to PATH mismatch
packages = [
    'gdown',
    'scikit-learn',
    'scikit-image',
    'matplotlib',
    'Pillow',
    'numpy',
]

print(f'Installing into: {sys.executable}\n')
!{sys.executable} -m pip install -q {' '.join(packages)}
print('\nInstallation complete.')

Installing into: /venv/main/bin/python


Installation complete.


In [ ]:
import sys

# Uncomment if torch / torchvision are missing on this instance
# (Vast.ai PyTorch templates have them pre-installed — check the GPU Check cell above)

# !{sys.executable} -m pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118

In [5]:
# Verify every package the project needs is importable in THIS kernel
import importlib, sys

required = {
    'torch'       : 'torch',
    'torchvision' : 'torchvision',
    'numpy'       : 'numpy',
    'PIL'         : 'Pillow',
    'sklearn'     : 'scikit-learn',
    'skimage'     : 'scikit-image',
    'matplotlib'  : 'matplotlib',
    'gdown'       : 'gdown',
}

all_ok = True
for mod, pkg in required.items():
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, '__version__', 'n/a')
        print(f'  OK  {pkg:<16} ({ver})')
    except ImportError:
        print(f'  MISSING  {pkg}  <-- run the install cell again')
        all_ok = False

print()
if all_ok:
    print('All packages verified — ready to continue.')
else:
    print('Fix missing packages before proceeding.')
    print('If the install cell ran but packages still show MISSING, restart the kernel')
    print('and run ONLY the install cell again before this one.')

  OK  torch            (2.11.0+cu128)
  OK  torchvision      (0.26.0+cu128)
  OK  numpy            (2.4.3)
  OK  Pillow           (12.1.1)
  OK  scikit-learn     (1.8.0)
  OK  scikit-image     (0.26.0)
  OK  matplotlib       (3.10.8)
  OK  gdown            (6.0.0)

All packages verified — ready to continue.


## 2. Clone / Upload Project Code

Choose **one** of the two options below depending on how your code is stored.

In [6]:
# ── OPTION A: Clone from GitHub (recommended) ──────────────────────────────
import os, sys

REPO_URL    = 'https://github.com/husseinmleng/SAR-Research.git'  # <-- change this
PROJECT_DIR = '/workspace/SAR_Dr.Ahmed'

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print('Directory already exists — pulling latest changes')
    !git -C {PROJECT_DIR} pull

os.chdir(PROJECT_DIR)

# Add project root to the kernel's module search path so local .py files are importable
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print('Working directory :', os.getcwd())
print('sys.path[0]       :', sys.path[0])

Directory already exists — pulling latest changes
Already up to date.
Working directory : /workspace/SAR_Dr.Ahmed
sys.path[0]       : /workspace/SAR_Dr.Ahmed


In [10]:
DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1KwQSL4PO6ARktlUc6F-cHNzSlTOuoMX6"                                     
                                                                                                                      
DATA_ROOT = "mstar_sampled_data"   # destination name on the instance                                               
#https://drive.google.com/drive/folders/1KwQSL4PO6ARktlUc6F-cHNzSlTOuoMX6?usp=drive_link                                                                                                                  # 
# ── Download only if not already present ───────────────────────────────────                                       
if os.path.isdir(DATA_ROOT) and any(os.scandir(DATA_ROOT)):                                                       
  print(f"✓ '{DATA_ROOT}' already exists — skipping download.")                                                   
else:                                                                                                               
  print("Downloading dataset folder from Google Drive …")                                                         
  # gdown.download_folder downloads the entire folder recursively                                                 
  # output= sets the parent directory; the folder itself keeps its Drive name                                     
  gdown.download_folder(                                                                                          
      url=DRIVE_FOLDER_URL,                                                                                       
      output=".",          # downloads into current directory                                                     
      quiet=False,                                                                                                
      use_cookies=False,                                                                                          
  )
  # If Drive folder is named differently than DATA_ROOT, rename it:                                               
  # (change 'mstar_sampled_data_drive_name' to whatever the Drive folder is called)                               
  import shutil; shutil.move("drive_folder_name", DATA_ROOT)                                                    
                                                                                                                  
  print("Done.")                                                                                                  
                                                                                                                
# ── Sanity check ────────────────────────────────────────────────────────────                                      
classes = sorted(os.listdir(DATA_ROOT))                                                                           
print(f"\nFound {len(classes)} class folders: {classes}")                                                           
for cls in classes:
  n = len(os.listdir(os.path.join(DATA_ROOT, cls)))                                                               
  print(f"  {cls}: {n} images")                                            

Retrieving folder contents


Retrieving folder 1EZ97CLA207r5bDzM9Ie36WsMtKwInk-I 2S1
Processing file 1gPObTkyAbAR752mRa_GI_rLdDg7I4IK_ hb14931.jpeg
Processing file 1jlGNAhPzoaz8ymLczA2FqLXT8Pp8h6zJ hb14932.jpeg
Processing file 1ec0MkjSDMB-2gO85Bg5nci-J33fEzOSl hb14933.jpeg
Processing file 1rGtkuvKvVUNfX-U6yBv8evz4ndq-uM5M hb14934.jpeg
Processing file 1ZW5QGJGWJxx8yi8R98o7DKMlROG7t7bP hb14935.jpeg
Processing file 1EyUKt0ooTs_q7azEG3QsbHx8UyaEULqa hb14936.jpeg
Processing file 1so-dQLnIp2mBGAtopJb1mi7VT168RlNA hb14937.jpeg
Processing file 10s_HLVWeHHC-MYiusBRtQzuNPxLJnFaX hb14938.jpeg
Processing file 1tiZsE1J76K3QxTRy_lvxgxHUBz1lwsN7 hb14939.jpeg
Processing file 1BG1zxQeTApKt3Ivk2RP4GfkAWIHyExYR hb14940.jpeg
Processing file 1Ss36Z-bSijI2_S__tq-TNs65PBGXBvJP hb14941.jpeg
Processing file 1VMXitKgGSqhwEsf3tKpy1_0jj5AiR3LF hb14942.jpeg
Processing file 1E1Gj6_tuEfIBWRBw9sxOdmjlRL_TEuQ0 hb14943.jpeg
Processing file 1ZImQ319p0SKOXPpl6dV1J5mmIQZQE6Tv hb14944.jpeg
Processing file 1uMTvhWlGzVvsXjKwm293bzVqlUp4Q4Q- hb14945.jpeg

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1gPObTkyAbAR752mRa_GI_rLdDg7I4IK_
To: /workspace/SAR_Dr.Ahmed/2S1/hb14931.jpeg
100%|██████████| 5.77k/5.77k [00:00<00:00, 1.66MB/s]
Downloading...
From: https://drive.google.com/uc?id=1jlGNAhPzoaz8ymLczA2FqLXT8Pp8h6zJ
To: /workspace/SAR_Dr.Ahmed/2S1/hb14932.jpeg
100%|██████████| 5.58k/5.58k [00:00<00:00, 5.14MB/s]
Downloading...
From: https://drive.google.com/uc?id=1ec0MkjSDMB-2gO85Bg5nci-J33fEzOSl
To: /workspace/SAR_Dr.Ahmed/2S1/hb14933.jpeg
100%|██████████| 6.01k/6.01k [00:00<00:00, 1.42MB/s]
Downloading...
From: https://drive.google.com/uc?id=1rGtkuvKvVUNfX-U6yBv8evz4ndq-uM5M
To: /workspace/SAR_Dr.Ahmed/2S1/hb14934.jpeg
100%|██████████| 6.23k/6.23k [00:00<00:00, 1.86MB/s]
Downloading...
From: https://drive.google.com/uc?id=1ZW5QGJGWJxx8yi8R98o7DKMlROG7t7bP
To: /workspace/SAR_Dr.Ahmed/2S1/hb14935.jpeg
100%|██████████| 6.10k/6.10k

FileURLRetrievalError: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=15hRDIOGshthS3zVkh6VKTrtt3kiDgYD9

but Gdown can't. Please check connections and permissions.

## 3. Download MSTAR Dataset from Google Drive

### How to get your Google Drive file ID

1. Upload `mstar_sampled_data.zip` to your Google Drive
2. Right-click → **Share** → **Anyone with the link**
3. Copy the link: `https://drive.google.com/file/d/1aBcDeFgHiJkLmNoPqRsTuVwXyZ/view`
4. The file ID is the part between `/d/` and `/view`: `1aBcDeFgHiJkLmNoPqRsTuVwXyZ`
5. Paste it into `DATASET_GDRIVE_ID` below

In [ ]:
import os, gdown, zipfile

DATASET_GDRIVE_ID = 'PASTE_YOUR_DATASET_FILE_ID_HERE'  # <-- change this
DATASET_ZIP       = '/workspace/mstar_sampled_data.zip'
DATASET_DIR       = os.path.join(os.getcwd(), 'mstar_sampled_data')

if os.path.exists(DATASET_DIR):
    print(f'Dataset already exists: {DATASET_DIR}')
    print('Delete it and re-run this cell to force a fresh download.')
else:
    print('Downloading dataset from Google Drive...')
    gdown.download(id=DATASET_GDRIVE_ID, output=DATASET_ZIP, quiet=False)
    print('Extracting...')
    with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
        z.extractall(os.getcwd())
    os.remove(DATASET_ZIP)
    print('Done.')

In [ ]:
# Verify dataset structure
import os

DATASET_DIR = os.path.join(os.getcwd(), 'mstar_sampled_data')

if not os.path.exists(DATASET_DIR):
    raise FileNotFoundError(f'Dataset not found at {DATASET_DIR} — run the download cell first')

classes = sorted(os.listdir(DATASET_DIR))
print(f'Found {len(classes)} classes:\n')
total = 0
for cls in classes:
    cls_path = os.path.join(DATASET_DIR, cls)
    if not os.path.isdir(cls_path):
        continue
    n = len(os.listdir(cls_path))
    total += n
    marker = '  [UNSEEN - held out]' if cls == 'T72' else ''
    print(f'  {cls:<15} {n:>4} images{marker}')

print(f'\nTotal images : {total}')
assert 'T72' in classes, 'T72 class not found — check dataset structure'
print('\nDataset structure OK')

In [ ]:
# Preview one image per class
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os, random

DATASET_DIR = os.path.join(os.getcwd(), 'mstar_sampled_data')
classes = sorted([c for c in os.listdir(DATASET_DIR)
                  if os.path.isdir(os.path.join(DATASET_DIR, c))])

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('MSTAR Dataset — One sample per class', fontsize=13)

for ax, cls in zip(axes.flat, classes):
    cls_dir = os.path.join(DATASET_DIR, cls)
    img_file = random.choice(os.listdir(cls_dir))
    img = mpimg.imread(os.path.join(cls_dir, img_file))
    ax.imshow(img, cmap='gray')
    ax.set_title(cls + (' [UNSEEN]' if cls == 'T72' else ''), fontsize=9)
    ax.axis('off')

plt.tight_layout()
os.makedirs('results', exist_ok=True)
plt.savefig('results/dataset_preview.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Create Output Directories

In [ ]:
import os
for d in ['model', 'log', 'results', 'gan_output', 'model/gan']:
    os.makedirs(d, exist_ok=True)
print('Output directories ready')

## 5. Shape Smoke Test

Verifies CNN and GNN produce the correct tensor shapes before committing to a long training run.

In [ ]:
import sys, os
# Ensure project root is on the path (safe to call multiple times)
PROJECT_DIR = os.getcwd()
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import torch
from cnn import EmbeddingCNN
from gnn import GNN_module

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running smoke test on: {device}\n')

# CNN: (batch=4, channels=1, H=100, W=100) -> (batch, 64)
cnn = EmbeddingCNN(image_size=100, cnn_feature_size=64,
                   cnn_hidden_dim=32, cnn_num_layers=4).to(device)
x   = torch.rand(4, 1, 100, 100, device=device)
out = cnn(x)
assert out.shape == (4, 64), f'CNN shape error: {out.shape}'
print(f'CNN output shape  : {list(out.shape)}  OK')

# GNN: 3-way 20-shot, batch=8
nway, shots, batch = 3, 20, 8
D     = 64 + nway + 1       # CNN embed + one-hot label
N     = nway * shots + 1    # support nodes + 1 query
nodes = torch.rand(batch, N, D, device=device)
gnn   = GNN_module(nway=nway, input_dim=D, hidden_dim=16).to(device)
logits = gnn(nodes)
assert logits.shape == (batch, nway + 1), f'GNN shape error: {logits.shape}'
print(f'GNN output shape  : {list(logits.shape)}  OK  (batch x C+1 logits)')

# DataLoader
from data import self_DataLoader
dl         = self_DataLoader(root='mstar_sampled_data', nway=3,
                             unseen_class='T72', unseen_ratio=1.0)
batch_data = dl.load_tr_batch(batch_size=4, nway=3, num_shots=5)
print(f'DataLoader batch  : {[list(b.shape) for b in batch_data[:4]]}  OK')

print('\nAll smoke tests PASSED — safe to train')

---
## 6. Experiment 1 — Paper Baseline: FSL + GNN (3-way 20-shot)

Reproduces the reference paper configuration.  
**Target**: seen ≥ 95%, unseen ≥ 62%, overall ≥ 85%  
**Expected runtime**: ~40 min on a single GPU

In [ ]:
import sys
!{sys.executable} main.py \
    --data_root     mstar_sampled_data \
    --model_type    gnn \
    --nway          3 \
    --shots         20 \
    --unseen_class  T72 \
    --unseen_ratio  1.0 \
    --batch_size    8 \
    --lr            0.001 \
    --max_iteration 100000 \
    --log_interval  100 \
    --eval_interval 500 \
    --early_stop    10 \
    --eval_output   results \
    --save \
    --use_gpu       0

In [ ]:
# Show last 30 lines of the training log
import glob
log_files = sorted(glob.glob('log/3way_20shot_gnn_/*_log.txt'))
if log_files:
    with open(log_files[0]) as f:
        lines = f.readlines()
    print(''.join(lines[-30:]))
else:
    print('Log file not found — check log/ directory')

## 7. Experiment 2 — CNN Baseline (Closed-Set)

Standard CNN classifier without few-shot learning — establishes the data-scarcity performance floor.  
**Expected runtime**: ~10 min each

In [ ]:
# Full-data CNN baseline
import sys
!{sys.executable} main.py \
    --data_root     mstar_sampled_data \
    --model_type    cnn \
    --nway          9 \
    --shots         20 \
    --batch_size    8 \
    --lr            0.001 \
    --max_iteration 50000 \
    --log_interval  100 \
    --eval_interval 500 \
    --early_stop    10 \
    --affix         full \
    --eval_output   results \
    --save \
    --use_gpu       0

In [ ]:
# K=10 shot CNN baseline (simulating data scarcity)
import sys
!{sys.executable} main.py \
    --data_root     mstar_sampled_data \
    --model_type    cnn \
    --nway          9 \
    --shots         10 \
    --baseline_kshot \
    --batch_size    8 \
    --lr            0.001 \
    --max_iteration 50000 \
    --log_interval  100 \
    --eval_interval 500 \
    --early_stop    10 \
    --affix         k10 \
    --eval_output   results \
    --save \
    --use_gpu       0

## 8. Experiment 3 — 5-way and 7-way Configurations

In [ ]:
import sys

for nway in [5, 7]:
    print(f'\n{"="*50}')
    print(f'Training {nway}-way 20-shot GNN')
    print(f'{"="*50}\n')
    !{sys.executable} main.py \
        --data_root     mstar_sampled_data \
        --model_type    gnn \
        --nway          {nway} \
        --shots         20 \
        --unseen_class  T72 \
        --unseen_ratio  1.0 \
        --batch_size    8 \
        --lr            0.001 \
        --max_iteration 100000 \
        --early_stop    10 \
        --eval_output   results \
        --save \
        --use_gpu       0

## 9. Experiment 4 — GAN Training and Augmented FSL

### Step 1: Train the Conditional DCGAN

Generates 100 synthetic SAR images per vehicle class.  
**Expected runtime**: ~60 min for 200 epochs on GPU

In [ ]:
import sys
!{sys.executable} gan.py \
    --data_root   mstar_sampled_data \
    --output_dir  gan_output \
    --epochs      200 \
    --batch_size  32 \
    --noise_dim   100 \
    --img_size    100 \
    --lr_g        0.0002 \
    --lr_d        0.0002 \
    --n_generate  100 \
    --seed        42 \
    --gpu         0

In [ ]:
# Verify GAN output
import os
gan_dir = 'gan_output'
if not os.path.exists(gan_dir):
    print('gan_output/ not found — run GAN training first')
else:
    for cls in sorted(os.listdir(gan_dir)):
        cls_path = os.path.join(gan_dir, cls)
        if os.path.isdir(cls_path):
            n = len(os.listdir(cls_path))
            status = 'OK' if n >= 100 else f'WARNING: only {n}'
            print(f'  {cls:<15} {n:>4} images  {status}')

In [ ]:
# Preview: real vs GAN-generated side-by-side
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os, random

classes_to_show = ['2S1', 'BMP2', 'T62']
fig, axes = plt.subplots(len(classes_to_show), 4, figsize=(10, 7))
fig.suptitle('Real (left 2 columns) vs GAN-generated (right 2 columns)', fontsize=11)

for row, cls in enumerate(classes_to_show):
    real_dir = os.path.join('mstar_sampled_data', cls)
    fake_dir = os.path.join('gan_output', cls)
    for col, (img_dir, label) in enumerate([
        (real_dir, 'Real'), (real_dir, 'Real'),
        (fake_dir, 'GAN'),  (fake_dir, 'GAN')
    ]):
        if not os.path.exists(img_dir) or not os.listdir(img_dir):
            axes[row, col].axis('off')
            continue
        img = mpimg.imread(os.path.join(img_dir, random.choice(os.listdir(img_dir))))
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].set_title(f'{label}\n{cls}', fontsize=8)
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('results/gan_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

### Step 2: FSL + GNN with GAN Augmentation (K=10)

In [ ]:
import sys
!{sys.executable} main.py \
    --data_root      mstar_sampled_data \
    --model_type     gnn \
    --nway           3 \
    --shots          10 \
    --unseen_class   T72 \
    --gan_augment \
    --gan_output_dir gan_output \
    --batch_size     8 \
    --lr             0.001 \
    --max_iteration  100000 \
    --early_stop     10 \
    --affix          gan_k10 \
    --eval_output    results \
    --save \
    --use_gpu        0

## 10. Experiment 5 — Physics-Informed Regularization

In [ ]:
import sys
!{sys.executable} main.py \
    --data_root      mstar_sampled_data \
    --model_type     gnn \
    --nway           3 \
    --shots          20 \
    --unseen_class   T72 \
    --physics_lambda 0.05 \
    --batch_size     8 \
    --lr             0.001 \
    --max_iteration  100000 \
    --early_stop     10 \
    --affix          physics \
    --eval_output    results \
    --save \
    --use_gpu        0

## 11. Experiment 6 — Rotation and Speckle Noise Invariance

In [ ]:
import sys
!{sys.executable} main.py \
    --data_root       mstar_sampled_data \
    --model_type      gnn \
    --nway            3 \
    --shots           20 \
    --unseen_class    T72 \
    --augment_rotation \
    --augment_speckle \
    --speckle_sigma   0.1 \
    --batch_size      8 \
    --lr              0.001 \
    --max_iteration   100000 \
    --early_stop      10 \
    --affix           augmented \
    --eval_output     results \
    --save \
    --use_gpu         0

## 12. Experiment 7 — Full Pipeline (All Enhancements)

GAN augmentation + physics regularization + rotation + speckle, at K=10.

In [ ]:
import sys
!{sys.executable} main.py \
    --data_root       mstar_sampled_data \
    --model_type      gnn \
    --nway            3 \
    --shots           10 \
    --unseen_class    T72 \
    --gan_augment \
    --gan_output_dir  gan_output \
    --physics_lambda  0.05 \
    --augment_rotation \
    --augment_speckle \
    --speckle_sigma   0.1 \
    --batch_size      8 \
    --lr              0.001 \
    --max_iteration   100000 \
    --early_stop      10 \
    --affix           full_pipeline \
    --eval_output     results \
    --save \
    --use_gpu         0

## 13. Evaluate a Saved Checkpoint (No Re-Training)

In [ ]:
import sys
!{sys.executable} main.py \
    --data_root    mstar_sampled_data \
    --model_type   gnn \
    --nway         3 \
    --shots        20 \
    --unseen_class T72 \
    --load \
    --load_dir     model/3way_20shot_gnn_ \
    --eval_only \
    --eval_output  results \
    --use_gpu      0

## 14. Results: Comparison Table and Charts

In [ ]:
import sys, os
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import json, glob
from evaluate import print_comparison_table

report_files = sorted(glob.glob('results/*.json'))

if not report_files:
    print('No result JSON files found yet — run training experiments first')
else:
    print(f'Found {len(report_files)} report(s):\n')
    for f in report_files:
        print(' ', f)
    print()
    table = print_comparison_table(report_files)
    with open('results/comparison_table.md', 'w') as f:
        f.write('# Results Comparison\n\n' + table)
    print('\nSaved to results/comparison_table.md')

In [ ]:
# Bar chart: overall / seen / unseen accuracy per experiment
import json, glob, os
import matplotlib.pyplot as plt

report_files = sorted(glob.glob('results/*.json'))

if report_files:
    configs, overall, seen, unseen = [], [], [], []
    for path in report_files:
        with open(path) as f:
            d = json.load(f)
        configs.append(d.get('config', os.path.basename(path).replace('.json', '')))
        overall.append(d.get('overall_accuracy', 0))
        seen.append(d.get('seen_accuracy', 0))
        unseen.append(d.get('unseen_accuracy', 0))

    x, w = range(len(configs)), 0.25
    fig, ax = plt.subplots(figsize=(max(8, len(configs) * 2), 5))
    ax.bar([i - w for i in x], overall, w, label='Overall', color='steelblue')
    ax.bar([i     for i in x], seen,    w, label='Seen',    color='seagreen')
    ax.bar([i + w for i in x], unseen,  w, label='Unseen',  color='tomato')
    ax.axhline(95, linestyle='--', color='seagreen', alpha=0.4, label='Seen target 95%')
    ax.axhline(62, linestyle='--', color='tomato',   alpha=0.4, label='Unseen target 62%')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Accuracy by Experiment')
    ax.set_xticks(list(x))
    ax.set_xticklabels(configs, rotation=30, ha='right', fontsize=8)
    ax.set_ylim(0, 110)
    ax.legend()
    plt.tight_layout()
    plt.savefig('results/accuracy_chart.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('No JSON reports found')

In [ ]:
# Confusion matrix for the 3-way 20-shot baseline
import json, glob
import numpy as np
import matplotlib.pyplot as plt

reports = glob.glob('results/3way_20shot_gnn_*.json')
if reports:
    with open(sorted(reports)[-1]) as f:
        d = json.load(f)

    cm = np.array(d['confusion_matrix'])
    names = list(d['per_class'].keys())
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title('Confusion Matrix — 3-way 20-shot GNN')
    for i in range(len(names)):
        for j in range(len(names)):
            v = cm_norm[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    fontsize=7, color='white' if v > 0.5 else 'black')
    plt.tight_layout()
    plt.savefig('results/confusion_matrix.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('No baseline report found — run Experiment 1 first')

## 15. Save Results to Google Drive

Run **one** of the three options below before your Vast.ai instance is terminated.

In [ ]:
# ── OPTION A: Vast.ai — zip everything and download via Jupyter file browser ─
import shutil, os, datetime

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
zip_path  = f'/workspace/sar_results_{timestamp}'
shutil.make_archive(zip_path, 'zip', os.getcwd())

final = zip_path + '.zip'
print(f'Archive: {final}  ({os.path.getsize(final)/1e6:.1f} MB)')
print('Download it from the Jupyter file browser (left panel)')

In [ ]:
# ── OPTION B: Vast.ai — sync to Google Drive via rclone ────────────────────
# First-time setup (run once in the Vast.ai terminal, NOT in this cell):
#   rclone config
#   --> choose Google Drive, name the remote 'gdrive'

import sys
GDRIVE_DEST = 'gdrive:SAR_results'  # <-- change folder name if needed

!rclone copy results/ {GDRIVE_DEST}/results/ --progress
!rclone copy model/   {GDRIVE_DEST}/model/   --progress
!rclone copy log/     {GDRIVE_DEST}/log/     --progress
print('Upload complete')

In [ ]:
# ── OPTION C: Google Colab only — mount Drive and copy ─────────────────────
from google.colab import drive
import shutil, os

drive.mount('/gdrive')
DRIVE_DEST = '/gdrive/MyDrive/SAR_results'  # <-- change if needed
os.makedirs(DRIVE_DEST, exist_ok=True)

for folder in ['results', 'log', 'model']:
    src = os.path.join(os.getcwd(), folder)
    if os.path.exists(src):
        shutil.copytree(src, os.path.join(DRIVE_DEST, folder), dirs_exist_ok=True)
        print(f'Copied {folder}/')
print('Done')

---
## Quick Reference

| Section | Experiment | Approx. GPU time |
|---------|-----------|------------------|
| §5  | Smoke test | < 1 min |
| §6  | FSL+GNN 3-way 20-shot (paper baseline) | ~40 min |
| §7  | CNN baseline full + K=10 | ~10 min each |
| §8  | 5-way + 7-way sweep | ~40 min each |
| §9  | GAN training 200 epochs | ~60 min |
| §9  | FSL+GNN + GAN augment K=10 | ~40 min |
| §10 | Physics loss λ=0.05 | ~40 min |
| §11 | Rotation + speckle augmentation | ~40 min |
| §12 | Full pipeline all enhancements | ~40 min |
| §13 | Eval-only on saved checkpoint | ~5 min |
| §14 | Comparison table + charts | < 1 min |

**Total**: ~6 hours for all experiments. Run §6 first to validate the paper baseline before continuing.

### Key output paths
```
model/3way_20shot_gnn_/model.pth     best checkpoint (Exp 1)
model/gan/generator.pt               trained GAN
results/*.json                       per-experiment metrics
results/comparison_table.md          printable results table
results/accuracy_chart.png           bar chart
results/confusion_matrix.png         confusion matrix
log/*/train_log.txt                  full training logs
```

### If you see ModuleNotFoundError after kernel restart
Re-run **§1 install cell** (the one with `sys.executable -m pip install`), then **§1 verify cell**. Do not use bare `!pip install` — it installs into a different Python than the kernel.